In [2]:
from itertools import count


def is_variable(x):
    return x.startswith("?")

def substitute(term, subs):
    while term in subs:
        term = subs[term]
    return term

def unify(x, y, subs):
    x = substitute(x, subs)
    y = substitute(y, subs)

    if x == y:
        return subs

    if is_variable(x):
        if x in y and is_variable(y):
            return None
        subs[x] = y
        return subs

    if is_variable(y):
        if y in x and is_variable(x):
            return None
        subs[y] = x
        return subs

    return None


def apply_substitution(literal, subs):
    neg, pred, args = literal
    args = tuple(substitute(a, subs) for a in args)
    return (neg, pred, args)


def standardize(clause):
    suffix = next(variable_counter)
    result = set()

    for neg, pred, args in clause:
        new_args = tuple(
            arg + "_" + str(suffix) if is_variable(arg) else arg
            for arg in args
        )
        result.add((neg, pred, new_args))

    return result


variable_counter = count()


def resolve(c1, c2):
    c1 = standardize(c1)
    c2 = standardize(c2)

    resolvents = []

    for l1 in c1:
        for l2 in c2:

            neg1, pred1, args1 = l1
            neg2, pred2, args2 = l2

            if neg1 == neg2 or pred1 != pred2:
                continue

            if len(args1) != len(args2):
                continue

            subs = {}
            valid = True


            for a, b in zip(args1, args2):
                if unify(a, b, subs) is None:
                    valid = False
                    break

            if valid:

                new_clause = (c1 - {l1}) | (c2 - {l2})
                new_clause = {
                    apply_substitution(lit, subs)
                    for lit in new_clause
                }

                resolvents.append(new_clause)

    return resolvents


def resolution(clauses):
    clauses = [set(c) for c in clauses]

    print("Initial Clauses:")
    for c in clauses:
        print(c)

    while True:
        new_clauses = []

        for i in range(len(clauses)):
            for j in range(i + 1, len(clauses)):

                resolvents = resolve(clauses[i], clauses[j])

                for clause in resolvents:


                    if len(clause) == 0:
                        print("\nEmpty clause found!")
                        print("Conclusion is PROVED.")
                        return True

                    if clause not in clauses and clause not in new_clauses:
                        new_clauses.append(clause)

        if not new_clauses:
            print("\nNo more resolution possible.")
            print("Conclusion could not be proved.")
            return False

        clauses.extend(new_clauses)

        print("\nNew Clauses:")
        for c in new_clauses:
            print(c)



C1 = {
    (True, "Human", ("?x",)),
    (False, "Mortal", ("?x",))
}

C2 = {
    (False, "Human", ("Socrates",))
}


C3 = {
    (True, "Mortal", ("Socrates",))
}


clauses = [C1, C2, C3]


resolution(clauses)


Initial Clauses:
{(False, 'Mortal', ('?x',)), (True, 'Human', ('?x',))}
{(False, 'Human', ('Socrates',))}
{(True, 'Mortal', ('Socrates',))}

New Clauses:
{(False, 'Mortal', ('Socrates',))}
{(True, 'Human', ('Socrates',))}

Empty clause found!
Conclusion is PROVED.


True